# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [4]:
# Write your code below.

%load_ext dotenv
%dotenv 

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [7]:
import dask.dataframe as dd

/tmp/ipykernel_115308/676544213.py:1: DeprecationWarning: The current Dask DataFrame implementation is deprecated. 
In a future release, Dask DataFrame will use a new implementation that
contains several improvements including a logical query planning.
The user-facing DataFrame API will remain unchanged.

The new implementation is already available and can be enabled by
installing the dask-expr library:

    $ pip install dask-expr

and turning the query planning option on:

    >>> import dask
    >>> dask.config.set({'dataframe.query-planning': True})
    >>> import dask.dataframe as dd

API documentation for the new implementation is available at
https://docs.dask.org/en/stable/dask-expr-api.html

Any feedback can be reported on the Dask issue tracker
https://github.com/dask/dask/issues 

To disable this warning in the future, set dask config:

    # via Python
    >>> dask.config.set({'dataframe.query-planning-warning': False})

    # via CLI
    dask config set dataframe.query-pla

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [8]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")
print("PRICE_DATA is:", PRICE_DATA)
stock_files = glob(os.path.join(PRICE_DATA,"*","*","*.parquet"), recursive = True)

PRICE_DATA is: ../../05_src/data/prices/


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [9]:
# Write your code below.
# Read all parquet files
dd_prices = dd.read_parquet(stock_files)

# Add lags for Close and Adj_Close
dd_prices = dd_prices.assign(
    Close_lag_1 = dd_prices["Close"].shift(1),
    Adj_Close_lag_1 = dd_prices["Adj Close"].shift(1)
)

# Returns based on Close
dd_prices = dd_prices.assign(
    returns = (dd_prices["Close"] / dd_prices["Close_lag_1"]) - 1
)

# Day's High minus Low
dd_prices = dd_prices.assign(
    hi_lo_range = dd_prices["High"] - dd_prices["Low"]
)

# Assign the result
dd_feat = dd_prices

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [10]:
# Write your code below.
# 1. Compute the full Dask DataFrame into pandas
fDD = dd_feat.compute()

# 2. Sort by ticker and Date
fDD = fDD.sort_values(["ticker", "Date"])

# 3. Window of 10 days
fDD["ma10_returns"] = fDD.groupby("ticker")["returns"].transform(lambda x: x.rolling(10).mean())

print(fDD.head(30))


             Date  Open   High    Low  Close  Adj Close    Volume   source  \
211892 1973-02-21   0.0  3.875  3.875  3.875   2.578557   22600.0  AEM.csv   
211893 1973-02-22   0.0  3.875  3.875  3.875   2.578557   56100.0  AEM.csv   
211894 1973-02-23   0.0  3.875  3.875  3.875   2.578557   42200.0  AEM.csv   
211895 1973-02-26   0.0  3.750  3.750  3.750   2.495378   30300.0  AEM.csv   
211896 1973-02-27   0.0  3.750  3.750  3.750   2.495378   16300.0  AEM.csv   
211897 1973-02-28   0.0  3.750  3.750  3.750   2.495378   11500.0  AEM.csv   
211898 1973-03-01   0.0  4.125  4.125  4.125   2.744915   40100.0  AEM.csv   
211899 1973-03-02   0.0  3.750  3.750  3.750   2.495378  108100.0  AEM.csv   
211900 1973-03-05   0.0  3.875  3.875  3.875   2.578557   13600.0  AEM.csv   
211901 1973-03-06   0.0  3.625  3.625  3.625   2.412198   12700.0  AEM.csv   
211902 1973-03-07   0.0  3.750  3.750  3.750   2.495378   14800.0  AEM.csv   
211903 1973-03-08   0.0  3.875  3.875  3.875   2.578557    9800.

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

It depends on the data size. If the data size is small, it is better to convert to pandas since it is simpler. But if the data is large and can not put directly into memory, we can use Dask. I is observed that using Dask is trickier than using Pandas.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.